# 01 — Exploratory Data Analysis: Media Bias in News Headlines

This notebook covers:
1. Load dataset and preview
2. Class distribution
3. Headline length by class
4. Top words per class
5. Word co-occurrence / association analysis
6. WordCloud per bias class
7. Bigram analysis per class

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
import sys, os
sys.path.insert(0, os.path.join('..', 'src'))
from preprocessing import preprocess_dataframe, get_vocab_stats

sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 120

df = pd.read_csv('../data/sample_headlines.csv')
print(f'Dataset shape: {df.shape}')
df.head(10)

## 1. Class Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

label_counts = df['label'].value_counts()
colors = ['#2166ac', '#4dac26', '#d01c8b']

# Bar chart
axes[0].bar(label_counts.index, label_counts.values, color=colors, edgecolor='black', linewidth=0.8)
axes[0].set_title('Class Distribution — News Headlines', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Bias Label')
axes[0].set_ylabel('Count')
for i, (label, count) in enumerate(label_counts.items()):
    axes[0].text(i, count + 0.3, str(count), ha='center', fontsize=12, fontweight='bold')

# Pie chart
axes[1].pie(label_counts.values, labels=[l.capitalize() for l in label_counts.index],
            colors=colors, autopct='%1.1f%%', startangle=90, textprops={'fontsize': 12})
axes[1].set_title('Class Proportions', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.savefig('../models/class_distribution.png', bbox_inches='tight')
plt.show()

print('Class distribution:')
print(label_counts)
print(f'\nBalance ratio (min/max): {label_counts.min()/label_counts.max():.2f}')

## 2. Preprocessing

In [ ]:
df_proc = preprocess_dataframe(df, text_col='headline')
stats = get_vocab_stats(df_proc)
print('Vocabulary Statistics:')
for k, v in stats.items():
    print(f'  {k}: {v:.1f}' if isinstance(v, float) else f'  {k}: {v}')
df_proc[['headline', 'cleaned', 'no_stopwords']].head(5)

## 3. Headline Length Distribution by Class

In [ ]:
df_proc['token_count'] = df_proc['tokens'].apply(len)
df_proc['char_count'] = df_proc['headline'].apply(len)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

label_color_map = {'left': '#2166ac', 'neutral': '#4dac26', 'right': '#d01c8b'}

# Token count by class
for label, color in label_color_map.items():
    subset = df_proc[df_proc['label'] == label]['token_count']
    axes[0].hist(subset, bins=8, alpha=0.6, color=color, label=label.capitalize(), edgecolor='black')
axes[0].set_title('Token Count Distribution by Class', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Number of Tokens')
axes[0].set_ylabel('Frequency')
axes[0].legend()

# Box plot
data_by_class = [df_proc[df_proc['label'] == l]['token_count'].values for l in ['left', 'neutral', 'right']]
bp = axes[1].boxplot(data_by_class, labels=['Left', 'Neutral', 'Right'], patch_artist=True, notch=False)
for patch, color in zip(bp['boxes'], ['#2166ac', '#4dac26', '#d01c8b']):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)
axes[1].set_title('Token Count Boxplot by Class', fontsize=13, fontweight='bold')
axes[1].set_ylabel('Token Count')

plt.tight_layout()
plt.show()

print('\nMean token count by class:')
print(df_proc.groupby('label')['token_count'].mean().round(2))

## 4. Top Words Per Class

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
labels_order = ['left', 'neutral', 'right']
colors_order = ['#2166ac', '#4dac26', '#d01c8b']

for ax, label, color in zip(axes, labels_order, colors_order):
    tokens = [t for tokens_list in df_proc[df_proc['label'] == label]['tokens'] for t in tokens_list
              if t.isalpha() and len(t) > 2]
    top_words = Counter(tokens).most_common(15)
    words, counts = zip(*top_words)
    ax.barh(words[::-1], counts[::-1], color=color, alpha=0.85, edgecolor='black', linewidth=0.5)
    ax.set_title(f'Top Words — {label.capitalize()}', fontsize=13, fontweight='bold')
    ax.set_xlabel('Frequency')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

plt.suptitle('Most Frequent Words by Bias Class (after stopword removal)', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## 5. Word Co-occurrence / Association Analysis

In [ ]:
from itertools import combinations

def compute_cooccurrence(tokens_series, top_n=20):
    """Compute word co-occurrence counts."""
    cooc = Counter()
    for tokens in tokens_series:
        clean = [t for t in tokens if t.isalpha() and len(t) > 2]
        for pair in combinations(set(clean), 2):
            cooc[tuple(sorted(pair))] += 1
    return cooc.most_common(top_n)

print('Top word co-occurrences by class:\n')
for label in ['left', 'neutral', 'right']:
    subset = df_proc[df_proc['label'] == label]['tokens']
    top_cooc = compute_cooccurrence(subset, top_n=5)
    print(f'  {label.upper()}:')
    for (w1, w2), count in top_cooc:
        print(f'    "{w1}" + "{w2}": {count}')
    print()

In [ ]:
# Co-occurrence heatmap for full vocabulary
all_tokens = df_proc['tokens'].apply(lambda t: [x for x in t if x.isalpha() and len(x) > 2])
all_top = [w for w, _ in Counter([t for tl in all_tokens for t in tl]).most_common(20)]

cooc_matrix = pd.DataFrame(0, index=all_top, columns=all_top)
for tokens in all_tokens:
    clean = [t for t in tokens if t in all_top]
    for w1, w2 in combinations(clean, 2):
        cooc_matrix.loc[w1, w2] += 1
        cooc_matrix.loc[w2, w1] += 1

plt.figure(figsize=(12, 10))
mask = np.eye(len(all_top), dtype=bool)
sns.heatmap(cooc_matrix, annot=True, fmt='d', cmap='YlOrRd', linewidths=0.3, mask=mask)
plt.title('Word Co-occurrence Heatmap (Top 20 Words)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 6. WordCloud Per Bias Class

In [ ]:
try:
    from wordcloud import WordCloud

    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    wc_colors = {'left': 'Blues', 'neutral': 'Greens', 'right': 'RdPu'}

    for ax, (label, cmap) in zip(axes, wc_colors.items()):
        text = ' '.join(df_proc[df_proc['label'] == label]['no_stopwords'])
        wc = WordCloud(
            width=600, height=400, background_color='white',
            colormap=cmap, max_words=60, prefer_horizontal=0.9
        ).generate(text)
        ax.imshow(wc, interpolation='bilinear')
        ax.set_title(f'WordCloud — {label.capitalize()}', fontsize=14, fontweight='bold')
        ax.axis('off')

    plt.suptitle('Word Clouds by Bias Label', fontsize=16, fontweight='bold')
    plt.tight_layout()
    plt.show()
except ImportError:
    print('Install wordcloud: pip install wordcloud')

## 7. Bigram Analysis Per Class

In [ ]:
def get_bigrams(tokens_series, top_n=10):
    bigrams = Counter()
    for tokens in tokens_series:
        clean = [t for t in tokens if t.isalpha() and len(t) > 2]
        for bg in zip(clean[:-1], clean[1:]):
            bigrams[bg] += 1
    return bigrams.most_common(top_n)

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

for ax, label, color in zip(axes, labels_order, colors_order):
    subset = df_proc[df_proc['label'] == label]['tokens']
    top_bigrams = get_bigrams(subset, top_n=10)
    bigram_labels = [f'{w1} + {w2}' for (w1, w2), _ in top_bigrams]
    bigram_counts = [c for _, c in top_bigrams]
    ax.barh(bigram_labels[::-1], bigram_counts[::-1], color=color, alpha=0.85, edgecolor='black', linewidth=0.5)
    ax.set_title(f'Top Bigrams — {label.capitalize()}', fontsize=12, fontweight='bold')
    ax.set_xlabel('Count')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

plt.suptitle('Top Bigrams by Bias Class', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## Summary

**Key EDA Findings:**
- Left-leaning headlines frequently use emotionally charged terms like *crisis*, *radical*, *crushed*, *record profits*
- Right-leaning headlines tend to use words like *Biden*, *liberal*, *radical*, *open border*, *crime*
- Neutral headlines have shorter, more factual phrasing: *votes*, *passes*, *rises*, *announces*
- Average headline length is similar across classes (~8–10 tokens)
- Strong lexical differentiation supports feasibility of automated bias classification